# Bracket matching — 01: detection and causal ablation

Identify which attention heads are responsible for SantaCoder's bracket matching
behavior, using the same two-pronged method as the induction-head work
(`scripts/benchmark.py`):

1. **Detection** — which heads attend from the final (query) position, where the closer
   should be predicted, back to the pending **opener** token.
2. **Causal ablation** — does zero-ablating a head reduce the closer logit difference?

A key finding from the induction stage is that the top *detection* head is not necessarily
the top *causal* head; we test for the same dissociation here. The prompt dataset is copied
verbatim from `00_setup.ipynb` so this notebook runs independently.

In [1]:
import json
import statistics
import sys
from pathlib import Path

import torch
from transformer_lens import HookedTransformer
from transformer_lens import utils as tl_utils

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.patch_santacoder import ensure_santacoder_transformers_compat

torch.set_grad_enabled(False)

MODEL_NAME = "santacoder"
DEVICE = "cpu"

for line in ensure_santacoder_transformers_compat():
    print(line)

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()
print(f"loaded {MODEL_NAME}: {model.cfg.n_layers} layers x {model.cfg.n_heads} heads")

already compatible: /home/estevao/.cache/huggingface/hub/models--bigcode--santacoder/snapshots/bb3be599767d93ce716293e9193c027e855a9524/modeling_gpt2_mq.py
synced snapshot to dynamic module cache: /home/estevao/.cache/huggingface/modules/transformers_modules/bigcode/santacoder/bb3be599767d93ce716293e9193c027e855a9524/modeling_gpt2_mq.py
already compatible: /home/estevao/.cache/huggingface/modules/transformers_modules/bigcode/santacoder/bb3be599767d93ce716293e9193c027e855a9524/modeling_gpt2_mq.py


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model santacoder into HookedTransformer
loaded santacoder: 24 layers x 16 heads


## Shared prompt dataset (copied from `00_setup.ipynb`)

In [2]:
BRACKET_PROMPTS = [
    # --- round parens ) ---
    {"code": "def foo(x, y", "closer": ")"},
    {"code": "result = max(first, second", "closer": ")"},
    {"code": "total = sum(values", "closer": ")"},
    {"code": "if (x > 0", "closer": ")"},
    {"code": "print(count, total", "closer": ")"},
    # --- square brackets ] ---
    {"code": "first = items[0", "closer": "]"},
    {"code": "cell = matrix[i][j", "closer": "]"},
    {"code": "data = [1, 2, 3", "closer": "]"},
    {"code": "last = arr[len(arr) - 1", "closer": "]"},
    {"code": "value = lookup[keys[0]", "closer": "]"},
    # --- curly braces } ---
    {"code": "if (x > 0) {", "closer": "}"},
    {"code": "const obj = {", "closer": "}"},
    {"code": "function add(a, b) {", "closer": "}"},
    {"code": "payload = {'key': value", "closer": "}"},
    {"code": "while (i < n) {", "closer": "}"},
]

CLOSERS = [")", "]", "}"]

def single_token_id(text):
    ids = model.to_tokens(text, prepend_bos=False)[0].tolist()
    assert len(ids) == 1, f"{text!r} is not a single token: {ids}"
    return ids[0]

closer_ids = {c: single_token_id(c) for c in CLOSERS}
print(f"{len(BRACKET_PROMPTS)} prompts; closer token ids: {closer_ids}")

15 prompts; closer token ids: {')': 8, ']': 60, '}': 92}


## Head detection: attention from the query position to the opener

For each prompt we locate the pending opener with a stack scan over characters (the
prompts contain no brackets inside string literals, so this is exact), map that character
to its token index via the fast tokenizer's offset mapping, and read the attention weight
from the final token (the closer-prediction site) to the opener token for every head.
Scores are averaged across prompts.

We keep `prepend_bos=True` consistently for both the forward pass and the index
bookkeeping (the BOS shifts every offset-derived index by one).

In [3]:
PREPEND_BOS = True

def pending_opener_char_index(code):
    """Char index of the last unmatched opener (no brackets inside strings)."""
    stack = []
    for i, ch in enumerate(code):
        if ch in "([{":
            stack.append(i)
        elif ch in ")]}":
            if stack:
                stack.pop()
    assert stack, f"no pending opener in {code!r}"
    return stack[-1]

def opener_query_positions(code):
    opener_char = pending_opener_char_index(code)
    enc = model.tokenizer(code, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    opener_tok = next((i for i, (a, b) in enumerate(offsets) if a <= opener_char < b), None)
    assert opener_tok is not None, f"could not locate opener token in {code!r}"
    bos = 1 if PREPEND_BOS else 0
    opener_pos = opener_tok + bos
    query_pos = len(offsets) + bos - 1
    return opener_pos, query_pos

detection = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
for p in BRACKET_PROMPTS:
    code = p["code"]
    tokens = model.to_tokens(code, prepend_bos=PREPEND_BOS)
    opener_pos, query_pos = opener_query_positions(code)
    assert query_pos == tokens.shape[1] - 1
    opener_char = code[pending_opener_char_index(code)]
    decoded = model.to_single_str_token(tokens[0, opener_pos].item())
    assert opener_char in decoded, f"opener mismatch {code!r}: token={decoded!r}"
    _, cache = model.run_with_cache(tokens)
    for layer in range(model.cfg.n_layers):
        pattern = cache["pattern", layer][0]  # [head, query, key]
        detection[layer] += pattern[:, query_pos, opener_pos]
detection /= len(BRACKET_PROMPTS)
print("detection scores computed")

detection scores computed


In [4]:
top = torch.topk(detection.flatten(), 10)
top_heads = []
for score, idx in zip(top.values.tolist(), top.indices.tolist()):
    layer = idx // model.cfg.n_heads
    head = idx % model.cfg.n_heads
    top_heads.append({"layer": layer, "head": head, "detection_score": score})

print("Top-10 heads by attention-to-opener score:")
for r in top_heads:
    print(f"  L{r['layer']}H{r['head']}   {r['detection_score']:.4f}")

Top-10 heads by attention-to-opener score:
  L14H10   0.3179
  L7H3   0.3066
  L5H1   0.3000
  L3H12   0.2791
  L3H2   0.2448
  L5H2   0.2421
  L3H9   0.2418
  L0H7   0.2359
  L17H8   0.2280
  L23H6   0.2192


## Causal ablation of the top-10 detected heads

Zero-ablate each head's value contribution (`z`) one at a time and re-measure the mean
closer logit difference. `causal_drop = clean - ablated`; a large positive drop means the
head is causally important for choosing the correct closer.

In [5]:
def mean_logit_diff(fwd_hooks=None):
    diffs = []
    for p in BRACKET_PROMPTS:
        tokens = model.to_tokens(p["code"], prepend_bos=PREPEND_BOS)
        if fwd_hooks:
            logits = model.run_with_hooks(tokens, fwd_hooks=fwd_hooks)
        else:
            logits = model(tokens)
        final = logits[0, -1]
        correct = final[closer_ids[p["closer"]]].item()
        best_other = max(final[closer_ids[c]].item() for c in CLOSERS if c != p["closer"])
        diffs.append(correct - best_other)
    return statistics.fmean(diffs)

def ablate_head_hook(head):
    def hook(z, hook):
        z[:, :, head, :] = 0.0
        return z
    return hook

clean = mean_logit_diff()
print(f"clean mean logit diff: {clean:+.4f}\n")

for r in top_heads:
    hooks = [(tl_utils.get_act_name("z", r["layer"]), ablate_head_hook(r["head"]))]
    ablated = mean_logit_diff(fwd_hooks=hooks)
    r["correct_logit_diff_clean"] = clean
    r["correct_logit_diff_ablated"] = ablated
    r["causal_drop"] = clean - ablated
    print(f"  L{r['layer']}H{r['head']}   ablated={ablated:+.4f}   drop={r['causal_drop']:+.4f}")

clean mean logit diff: +8.5342



  L14H10   ablated=+8.5320   drop=+0.0022


  L7H3   ablated=+8.5498   drop=-0.0157


  L5H1   ablated=+8.5415   drop=-0.0073


  L3H12   ablated=+8.5148   drop=+0.0194


...

  L5H2   ablated=+8.5523   drop=-0.0181


  L3H9   ablated=+8.5402   drop=-0.0060


  L0H7   ablated=+8.5534   drop=-0.0193


  L17H8   ablated=+8.5479   drop=-0.0137


  L23H6   ablated=+8.5544   drop=-0.0202


In [6]:
# Combined ranking: sum of detection rank and causal rank (lower = better on both).
det_order = sorted(top_heads, key=lambda r: r["detection_score"], reverse=True)
caus_order = sorted(top_heads, key=lambda r: r["causal_drop"], reverse=True)
det_rank = {(r["layer"], r["head"]): i for i, r in enumerate(det_order)}
caus_rank = {(r["layer"], r["head"]): i for i, r in enumerate(caus_order)}
for r in top_heads:
    key = (r["layer"], r["head"])
    r["detection_rank"] = det_rank[key]
    r["causal_rank"] = caus_rank[key]
    r["combined_rank_sum"] = det_rank[key] + caus_rank[key]

header = f"{'head':<8}{'detect':>9}{'ablated':>10}{'causaldrop':>12}{'detR':>6}{'causR':>7}"
print(header)
print("-" * len(header))
for r in sorted(top_heads, key=lambda r: r["combined_rank_sum"]):
    name = f"L{r['layer']}H{r['head']}"
    print(
        f"{name:<8}{r['detection_score']:>9.4f}{r['correct_logit_diff_ablated']:>10.3f}"
        f"{r['causal_drop']:>12.4f}{r['detection_rank']:>6d}{r['causal_rank']:>7d}"
    )

out_dir = ROOT / "outputs" / "bracket_matching"
out_dir.mkdir(parents=True, exist_ok=True)
payload = {
    "model_name": MODEL_NAME,
    "n_prompts": len(BRACKET_PROMPTS),
    "clean_mean_logit_diff": clean,
    "top_heads": top_heads,
}
out_path = out_dir / "detection_causal.json"
out_path.write_text(json.dumps(payload, indent=2))
print(f"\nsaved {out_path}")

head       detect   ablated  causaldrop  detR  causR
----------------------------------------------------
L14H10     0.3179     8.532      0.0022     0      2
L3H12      0.2791     8.515      0.0194     3      1
L3H2       0.2448     8.509      0.0254     4      0
L5H1       0.3000     8.541     -0.0073     2      4
L7H3       0.3066     8.550     -0.0157     1      6
L3H9       0.2418     8.540     -0.0060     6      3
L5H2       0.2421     8.552     -0.0181     5      7
L17H8      0.2280     8.548     -0.0137     8      5
L0H7       0.2359     8.553     -0.0193     7      8
L23H6      0.2192     8.554     -0.0202     9      9

saved /home/estevao/mechinterp/outputs/bracket_matching/detection_causal.json


## Interpretation

Read the table above against these questions:

- **Strongest causal contributors.** The heads with the largest positive `causal_drop` are
  the ones whose removal most damages closer selection — the primary candidates for a
  bracket-matching circuit. Heads whose drop is ~0 (or negative) attend to the opener but
  do not write the information that drives the decision.
- **Detection vs. causation dissociation.** Compare the `detR` and `causR` columns. If the
  #1 detection head is not the #1 causal head, we reproduce the central induction-stage
  finding: attention-pattern saliency alone is an unreliable guide to causal importance,
  which is exactly why ablation is run over the top-k detected heads rather than the single
  most-attending head.
- **Layer clustering.** Note the layers of the high-`causal_drop` heads. Matching a closer
  to its opener is a retrieval-then-copy operation; if the causal heads cluster in the
  mid-to-late layers (after positional/structural features are available) that is
  consistent with a dedicated matching mechanism rather than diffuse contribution.

**Next steps** suggested by the results: multi-head ablation of the top causal set (single
heads were materially weaker than head sets in the induction work), and per-closer-type
breakdown of the causal drops to see whether one mechanism serves all three bracket types
or whether `}` relies on different heads.

> Research input, not a finished claim: the evidence record is
> `outputs/bracket_matching/detection_causal.json`.

## Multi-head ablation

Single-head zero-ablation produced only small drops relative to the +8.53 baseline (best: **L3H2**, drop 0.025). If bracket matching is a **distributed** computation, ablating several causal heads *together* in one forward pass should produce a clearly larger effect than any head alone — the pattern seen in the induction-head stage, where head sets were materially stronger than single heads.

We test two groups against the clean baseline and the single best head:

- **two-head**: the two strongest individual contributors, `L3H2 + L3H12`;
- **five-head**: the top five by causal drop, `L3H2, L3H12, L3H9, L5H1, L17H8`.

All selected heads' `z` outputs are zeroed simultaneously, and we recompute the same metric (mean logit diff, correct closer vs. best incorrect closer).

In [7]:
def ablate_heads_hooks(heads):
    """Build forward hooks that zero the z-output of every (layer, head) in `heads`."""
    by_layer = {}
    for layer, head in heads:
        by_layer.setdefault(layer, []).append(head)
    hooks = []
    for layer, hs in by_layer.items():
        def hook(z, hook, hs=tuple(hs)):
            for h in hs:
                z[:, :, h, :] = 0.0
            return z
        hooks.append((tl_utils.get_act_name("z", layer), hook))
    return hooks

two_head_set = [(3, 2), (3, 12)]
five_head_set = [(3, 2), (3, 12), (3, 9), (5, 1), (17, 8)]

two_ablated = mean_logit_diff(fwd_hooks=ablate_heads_hooks(two_head_set))
five_ablated = mean_logit_diff(fwd_hooks=ablate_heads_hooks(five_head_set))

single_best = next(r for r in top_heads if (r["layer"], r["head"]) == (3, 2))

def fmt_heads(heads):
    return "+".join(f"L{l}H{h}" for l, h in heads)

summary = [
    {"label": "clean baseline", "heads": [],
     "ablated_mean_logit_diff": clean, "causal_drop": 0.0},
    {"label": "single best", "heads": [[3, 2]],
     "ablated_mean_logit_diff": single_best["correct_logit_diff_ablated"],
     "causal_drop": single_best["causal_drop"]},
    {"label": "two-head combined", "heads": [[l, h] for l, h in two_head_set],
     "ablated_mean_logit_diff": two_ablated, "causal_drop": clean - two_ablated},
    {"label": "five-head combined", "heads": [[l, h] for l, h in five_head_set],
     "ablated_mean_logit_diff": five_ablated, "causal_drop": clean - five_ablated},
]

header = f"{'configuration':<20}{'heads':<22}{'ablated diff':>14}{'causal drop':>14}"
print(header)
print("-" * len(header))
for row in summary:
    heads_str = fmt_heads([(l, h) for l, h in row["heads"]]) if row["heads"] else "(none)"
    print(f"{row['label']:<20}{heads_str:<22}{row['ablated_mean_logit_diff']:>14.4f}{row['causal_drop']:>14.4f}")

# Re-save the analysis JSON with the multi-head results appended.
payload = {
    "model_name": MODEL_NAME,
    "n_prompts": len(BRACKET_PROMPTS),
    "clean_mean_logit_diff": clean,
    "top_heads": top_heads,
    "multi_head_ablation": summary,
}
out_dir = ROOT / "outputs" / "bracket_matching"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "detection_causal.json"
out_path.write_text(json.dumps(payload, indent=2))
print(f"\nsaved {out_path}")

configuration       heads                   ablated diff   causal drop
----------------------------------------------------------------------
clean baseline      (none)                        8.5342        0.0000
single best         L3H2                          8.5088        0.0254
two-head combined   L3H2+L3H12                    8.4891        0.0450
five-head combined  L3H2+L3H12+L3H9+L5H1+L17H8        8.5238        0.0103

saved /home/estevao/mechinterp/outputs/bracket_matching/detection_causal.json


## Interpretation — multi-head ablation

| configuration | heads | causal drop |
|---|---|---|
| clean baseline | — | 0.000 |
| single best | L3H2 | 0.025 |
| two-head combined | L3H2 + L3H12 | 0.045 |
| five-head combined | L3H2, L3H12, L3H9, L5H1, L17H8 | 0.010 |

**Does combining heads produce a meaningfully larger effect?** Only for the two genuinely causal heads, and only mildly. Ablating `L3H2 + L3H12` together gives a drop of **0.045** — about double the single best head (0.025) and almost exactly the sum of their individual drops (0.025 + 0.019 = 0.044). So the two strongest heads are **roughly additive**; there is no strong synergy of the kind seen in the synthetic induction task. The **five-head** set is the cautionary result: its combined drop (**0.010**) is *smaller* than the single best head. The three heads added beyond the pair (`L3H9`, `L5H1`, `L17H8`) each had **negative** individual drops — zero-ablating them on their own slightly *helps* the metric — so pooling them with the two good heads partially cancels the effect. Naively ablating the "top-k by causal drop" is therefore the wrong move: which heads you include matters more than how many.

**Distributed or concentrated?** Even the best configuration (two-head, 0.045) is tiny against the **+8.53** clean baseline — about **0.5%** of the closer-selection signal. Zero-ablating a handful of the top detected heads barely dents the behavior. This points to a **highly distributed and redundant** mechanism: bracket matching in SantaCoder is not carried by a small isolated set of heads, and the top-10 detected heads capture only a sliver of it. A faithful circuit account would need to search well beyond this candidate pool (e.g. greedy/iterative head-set growth that actually re-scores the set after each addition, or path patching), rather than ranking heads independently and pooling the top few.

**Comparison to the induction-head stage.** The qualitative lessons transfer, but the magnitudes do not. In the induction stage, single-head zero-ablation was weak and a small head set was *materially* stronger — the synthetic set `L1H11 + L8H6 + L4H13` reached a causal selective drop of ~0.23 vs. ~0.08 for the best single head, i.e. clear cooperation. Here the multi-head benefit is real but modest (0.045 vs. 0.025) and shows no super-additive synergy, while the absolute effects are an order of magnitude smaller. Two findings do reproduce cleanly: (1) **detection ≠ causation** — the top detection head L14H10 has ~0 causal drop, while early-layer L3H2/L3H12 dominate causally; and (2) single-head attribution understates the mechanism. But the dominant story for this naturalistic task is **robustness**: the behavior is far more distributed and ablation-resistant than the synthetic induction probe.

> Research input, not a finished claim: the evidence record is `outputs/bracket_matching/detection_causal.json`.